   
# Lab: Content Recommendation Engine
## Your Step-by-Step Guide

This notebook walks you through building an **end-to-end content recommendation system** on Databricks — from raw viewing data all the way to a live REST API serving personalized picks in under 10 ms. Every step tells you **exactly which cell to run**, **what output to expect**, and **what to do if something goes wrong**.

---

### 🧪 This Is a Hands-On Lab

This notebook contains **6 coding exercises** where key parts of the code have been removed. Look for `<FILL_IN>` placeholders — replace them with the correct values to make each cell run successfully.

| Exercise | Section | Concept Tested |
|---|---|---|
| Exercise 1 | Section 2 | Engineer user-level features with PySpark aggregations |
| Exercise 2 | Section 2 | Build a point-in-time correct training interaction matrix |
| Exercise 3 | Section 3 | Configure ALS collaborative filtering model parameters |
| Exercise 4 | Section 3 | Register a trained model to Unity Catalog with MLflow |
| Exercise 5 | Section 4 | Create a Feature Spec for real-time serving |
| Exercise 6 | Section 4 | Query the Feature Serving endpoint |

### Architecture
```
Viewing History (Delta) → Feature Engineering → ALS Training (MLflow) → Model Registry (UC) → Feature Serving Endpoint
```

### Prerequisites
- Databricks workspace with Unity Catalog enabled
- Serverless compute or a cluster with `databricks-feature-engineering` installed
- The `00_Setup_Data_Generation` notebook must be in the same folder as this notebook

---
# Section 1: Setup & Data Exploration

**Goal:** Install required packages, generate synthetic data, and explore the viewing dataset to understand patterns before building features.

---

### Background

**What are we building?** A recommendation engine for a streaming platform that provides two types of suggestions:
1. **"Top 10 For You"** — personalized picks based on a user's viewing history
2. **"Because You Watched X"** — items similar to something the user just finished

**The data** is synthetic and models a streaming service with:
- **500 users** across different subscription tiers and regions
- **200 content items** spanning 10 genres (Drama, Comedy, Sci-Fi, etc.)
- **~50,000 viewing events** with ratings, watch completion %, and device info

### Step 1.1 — Install Required Packages

1. Run the cell directly below (**Shift + Enter**)
2. **What you should see:** Package installation output followed by a Python restart message
3. **Important:** Python restarts after this cell — any variables from before will be cleared. This is expected behavior

In [0]:
%pip install databricks-feature-engineering>=0.8.0 -q
dbutils.library.restartPython()

### Step 1.2 — Run the Data Generation Setup

1. Run the cell directly below (**Shift + Enter**)
2. **What you should see:** Output confirming that the `content_reco_lab` catalog, `content_reco_demo` schema, and three tables (`users`, `content_catalog`, `viewing_history`) were created with synthetic data
3. **This cell may take 1–2 minutes** to complete because it generates synthetic viewing data. Wait for it to finish before moving on

In [0]:
%run "./00_Setup_Data_Generation"

### Step 1.3 — Configuration and Imports

1. Run the cell directly below (**Shift + Enter**)
2. **What you should see:** Three lines confirming the schema, experiment, and model paths
3. **What it does:** Sets up all the constants and imports used throughout the lab. The `TRAINING_CUTOFF` date ensures point-in-time correctness (more on this in Section 2)

In [0]:
# --- Lab Configuration ---
CATALOG = "content_reco_lab"
SCHEMA = "content_reco_lab.content_reco_demo"
MODEL_NAME = f"{SCHEMA}.content_recommender_als"  # Unity Catalog model path
EXPERIMENT_NAME = "/Users/{}/content_reco_experiment".format(spark.sql("SELECT current_user()").first()[0])

# --- Imports ---
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
import mlflow, warnings, logging
from mlflow.models import infer_signature
import mlflow.spark
from datetime import datetime, timedelta

warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)

print(f"Schema: {SCHEMA}")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Model: {MODEL_NAME}")

### Step 1.4 — Load and Explore the Raw Data

1. Run the cell directly below (**Shift + Enter**)
2. **What you should see:**
   - A "Dataset Overview" with counts for Users (500), Content Items (200), and Viewing Events (~50,000)
   - Date range, average rating, and average watch completion %
   - A sample of 5 user records
3. **Take a moment to review the user table columns** — these fields (`subscription_tier`, `region`, `preferred_genres`, `age`) will become features later

In [0]:
# Load the three source tables
df_users = spark.table(f"{SCHEMA}.users")
df_content = spark.table(f"{SCHEMA}.content_catalog")
df_views = spark.table(f"{SCHEMA}.viewing_history")

print("Dataset Overview")
print("=" * 50)
print(f"  Users:           {df_users.count():>8,}")
print(f"  Content Items:   {df_content.count():>8,}")
print(f"  Viewing Events:  {df_views.count():>8,}")

# Show date range of viewing history
date_range = df_views.agg(
    F.min("view_timestamp").alias("earliest"),
    F.max("view_timestamp").alias("latest"),
    F.avg("rating").alias("avg_rating"),
    F.avg("watch_pct").alias("avg_watch_pct")
).first()

print(f"\n  Date Range:      {date_range.earliest} → {date_range.latest}")
print(f"  Avg Rating:      {date_range.avg_rating:.2f}")
print(f"  Avg Watch %:     {date_range.avg_watch_pct:.1f}%")

print("Sample Users:")
display(df_users.limit(5))

### Step 1.5 — Exploratory Data Analysis

1. Run the cell directly below (**Shift + Enter**)
2. **What you should see:** Two tables:
   - **Genre Popularity & Engagement** — total views, average rating, and completion % by genre
   - **Engagement by Subscription Tier** — how Free, Basic, and Premium users differ in viewing behavior
3. **Key insight to notice:** Premium users tend to have higher completion rates and views per user. Genre popularity will inform whether our model's recommendations align with actual preferences

In [0]:
# --- Exploratory Data Analysis ---
# Understand viewing patterns that will inform our feature engineering

# 1. Views by genre (which genres are most popular?)
df_genre_stats = (
    df_views
    .join(df_content, "content_id")
    .groupBy("genre")
    .agg(
        F.count("*").alias("total_views"),
        F.avg("rating").alias("avg_rating"),
        F.avg("watch_pct").alias("avg_completion"),
        F.countDistinct("user_id").alias("unique_viewers")
    )
    .orderBy(F.desc("total_views"))
)

print("Genre Popularity & Engagement:")
display(df_genre_stats)

# 2. Engagement by subscription tier
df_tier_stats = (
    df_views
    .join(df_users, "user_id")
    .groupBy("subscription_tier")
    .agg(
        F.count("*").alias("total_views"),
        F.avg("rating").alias("avg_rating"),
        F.avg("watch_pct").alias("avg_completion"),
        F.countDistinct("user_id").alias("active_users")
    )
    .withColumn("views_per_user", F.round(F.col("total_views") / F.col("active_users"), 1))
    .orderBy("subscription_tier")
)

print("Engagement by Subscription Tier:")
display(df_tier_stats)

> ✅ **Checkpoint:** You have installed the required packages, generated synthetic data, and explored the viewing dataset. You should now understand the data shape (500 users, 200 items, ~50K events) and key patterns (genre popularity, tier engagement). **Scroll down to continue to Section 2.**

---
# Section 2: Feature Engineering with Delta Lake

**Goal:** Transform raw viewing history into ML-ready features for users and content, then create a point-in-time correct training dataset for the ALS model.

---

### Background

**Why feature engineering matters:** Great recommendation systems depend on high-quality features. We need to capture each user's viewing behavior (what genres they prefer, how often they watch, how engaged they are) and each content item's characteristics (popularity, average ratings, completion rates).

**Point-in-time correctness:** When building features for training, you must ensure features are computed using **only data available at prediction time** — not future data. This prevents **data leakage** and ensures your model generalizes to production.

```
Training Data (before cutoff)     Test Data (after cutoff)
|<────────────────────────>|<───────────────────>|
                            2025-12-31
```

Follow the steps below in order.

### Step 2.1 — 🧪 Exercise 1: Engineer User-Level Features

> **This is a coding exercise.** Replace each `<FILL_IN>` placeholder with the correct value.

1. Read through the code in the cell below and replace each `<FILL_IN>` with the correct value
2. Run the cell (**Shift + Enter**)
3. **What you should see:** A confirmation message showing user features created for 500 users, followed by a preview of 5 user feature rows
4. **What it does:** Splits viewing history at the training cutoff, then computes per-user aggregations including engagement metrics, recency, genre diversity, binge behavior, and primary device

**Hints:**
* The first `<FILL_IN>` computes average watch completion — which column in the viewing data holds the watch completion percentage?
* The second `<FILL_IN>` computes average rating per user — which PySpark function calculates an average?
* The third `<FILL_IN>` finds the most recent viewing date per user — which PySpark aggregate function returns the maximum value?

In [0]:
# --- User-Level Feature Engineering ---
# These features capture each user's viewing behavior and preferences.
# In production, these would be maintained in the Databricks Feature Store.

# Define a "training cutoff" for point-in-time correctness
# We'll use data up to 2025-12-31 for features, hold out 2026+ for validation
TRAINING_CUTOFF = "2025-12-31"

df_views_train = df_views.filter(F.col("view_timestamp") <= TRAINING_CUTOFF)
df_views_test = df_views.filter(F.col("view_timestamp") > TRAINING_CUTOFF)

print(f"Training views (before {TRAINING_CUTOFF}): {df_views_train.count():,}")
print(f"Test views (after {TRAINING_CUTOFF}):      {df_views_test.count():,}")

# Enrich views with content metadata for feature engineering
df_views_enriched = df_views_train.join(df_content, "content_id")

# =============================================================================
# EXERCISE 1: Complete the user feature aggregations
# Replace each <FILL_IN> with the correct value.
# =============================================================================

# Step 1: Core user engagement features
df_user_features = (
    df_views_enriched
    .groupBy("user_id")
    .agg(
        # Engagement metrics
        F.count("*").alias("total_views"),
        F.countDistinct("content_id").alias("unique_content_viewed"),
        F.avg(<FILL_IN>).alias("avg_watch_completion"),            # Which column holds watch completion %?
        F.<FILL_IN>("rating").alias("avg_rating_given"),            # Which function computes an average?
        F.count(F.when(F.col("rating").isNotNull(), 1)).alias("num_ratings"),
        
        # Recency
        F.<FILL_IN>("view_timestamp").alias("last_view_date"),      # Which function gets the most recent date?
        F.datediff(F.lit(TRAINING_CUTOFF), F.max("view_timestamp")).alias("days_since_last_view"),
        
        # Genre diversity (how many distinct genres does the user watch?)
        F.countDistinct("genre").alias("genre_diversity")
    )
)

# Step 2: Binge indicator - max views per user per day
df_daily_views = (
    df_views_train
    .withColumn("view_date", F.to_date("view_timestamp"))
    .groupBy("user_id", "view_date")
    .agg(F.count("*").alias("daily_views"))
)

df_binge = (
    df_daily_views
    .groupBy("user_id")
    .agg(F.max("daily_views").alias("max_daily_views"))
)

# Step 3: Primary device (most frequently used device per user)
df_device_mode = (
    df_views_train
    .groupBy("user_id", "device_type")
    .agg(F.count("*").alias("device_count"))
)

w = Window.partitionBy("user_id").orderBy(F.desc("device_count"))
df_primary_device = (
    df_device_mode
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .select("user_id", F.col("device_type").alias("primary_device"))
)

# Combine all user features
df_user_features = (
    df_user_features
    .join(df_binge, "user_id", "left")
    .join(df_primary_device, "user_id", "left")
    .join(df_users.select("user_id", "subscription_tier", "region", "age"), "user_id")
)

print(f"\n\u2705 User features created for {df_user_features.count()} users")
display(df_user_features.limit(5))

### Step 2.2 — Engineer Content-Level Features

1. Run the cell directly below (**Shift + Enter**)
2. **What you should see:** A count of content features created for 200 items, followed by a table of the top 10 most-viewed content with title, genre, type, view count, average rating, and completion rate
3. **What it does:** Computes per-item popularity metrics (total views, unique viewers), quality signals (average rating, rating variance), and engagement depth (high completion vs. early dropout counts)

In [0]:
# --- Content-Level Feature Engineering ---
# These features capture how popular and well-received each content item is.

df_content_features = (
    df_views_train
    .groupBy("content_id")
    .agg(
        # Popularity metrics
        F.count("*").alias("total_views"),
        F.countDistinct("user_id").alias("unique_viewers"),
        
        # Quality signals
        F.avg("rating").alias("avg_user_rating"),
        F.stddev("rating").alias("rating_stddev"),
        F.avg("watch_pct").alias("avg_completion_rate"),
        
        # Engagement depth
        F.count(F.when(F.col("watch_pct") > 80, 1)).alias("high_completion_count"),
        F.count(F.when(F.col("watch_pct") < 20, 1)).alias("early_dropout_count")
    )
    .withColumn("completion_ratio",
        F.round(F.col("high_completion_count") / F.col("total_views"), 3)
    )
)

# Join with content metadata
df_content_features = (
    df_content_features
    .join(df_content, "content_id")
)

print(f"Content features created for {df_content_features.count()} items")
print("Top 10 Most Viewed Content:")
display(
    df_content_features
    .select("title", "genre", "content_type", "total_views", "avg_user_rating", "avg_completion_rate")
    .orderBy(F.desc("total_views"))
    .limit(10)
)

### Step 2.3 — 🧪 Exercise 2: Create the Point-in-Time Training Dataset

> **This is a coding exercise.** Replace each `<FILL_IN>` placeholder with the correct value.

1. Read through the code in the cell below and replace each `<FILL_IN>` with the correct value
2. Run the cell (**Shift + Enter**)
3. **What you should see:**
   - Training interaction matrix stats (unique user-content pairs, unique users, unique items, sparsity)
   - Confirmation that three feature tables were saved to Delta
   - A preview of 10 interaction rows
4. **What it does:** Creates the core input for ALS — a table of (user_id, content_id, rating) triplets from training-period data only

**Hints:**
* The first `<FILL_IN>` filters out rows where the rating is missing — which column name should you check for null?
* The second `<FILL_IN>` is the column that identifies which user gave the rating
* The third `<FILL_IN>` is the column that identifies which content item was rated
* The fourth `<FILL_IN>` averages the ratings when a user has watched the same content multiple times — which column holds the rating value?

In [0]:
# --- Point-in-Time Correct Training Dataset ---
# For collaborative filtering, our core input is the user-item interaction matrix.
# We use ONLY training-period data to prevent leakage.
#
# KEY CONCEPT: Point-in-time correctness means:
# - Features for user U are computed from views BEFORE the training cutoff
# - The target (rating/implicit feedback) is also from the training period
# - Test data (after cutoff) is reserved for evaluation only

# =============================================================================
# EXERCISE 2: Build the interaction matrix for ALS
# ALS needs: user_id (int), content_id (int), rating (float)
#
# Replace each <FILL_IN> with the correct value.
# =============================================================================

df_interactions = (
    df_views_train
    .filter(F.col(<FILL_IN>).isNotNull())       # Which column must not be null for explicit feedback?
    .groupBy(<FILL_IN>, <FILL_IN>)               # Group by which two ID columns?
    .agg(
        F.avg(<FILL_IN>).alias("rating"),         # Average which column if multiple views?
        F.count("*").alias("view_count"),
        F.avg("watch_pct").alias("avg_watch_pct")
    )
)

print(f"Training Interaction Matrix:")
print(f"Unique user-content pairs: {df_interactions.count():,}")
print(f"Unique users:             {df_interactions.select('user_id').distinct().count():,}")
print(f"Unique content items:     {df_interactions.select('content_id').distinct().count():,}")
print(f"Sparsity:                 {1 - df_interactions.count() / (500 * 200):.2%}")

# Save feature tables as Delta for reproducibility
df_user_features.write.mode("overwrite").saveAsTable(f"{SCHEMA}.user_features")
df_content_features.write.mode("overwrite").saveAsTable(f"{SCHEMA}.content_features")
df_interactions.write.mode("overwrite").saveAsTable(f"{SCHEMA}.training_interactions")

print(f"Feature tables saved to {SCHEMA}")
print("  → user_features")
print("  → content_features")
print("  → training_interactions")

display(df_interactions.limit(10))

> ✅ **Checkpoint:** You have engineered user-level and content-level features, and created a point-in-time correct training interaction matrix. All three feature tables are saved as Delta tables. **Scroll down to continue to Section 3.**

---
# Section 3: Model Training with ALS & MLflow

**Goal:** Train an ALS collaborative filtering model, tune hyperparameters with MLflow tracking, evaluate recommendations, and register the best model to Unity Catalog.

---

### What is ALS? (The Simple Version)

Imagine a giant spreadsheet where every **row is a user** and every **column is a show or movie**. Each cell holds that user's rating — but almost every cell is **blank**, because no one has time to watch everything!

Our data doesn't actually look like that giant grid — it's stored as a simple list of entries like *(User 42 rated Movie 7 → 4.5 stars)*. But ALS **thinks** of it as that big grid and tries to **fill in all the blanks**, predicting what rating each user *would* give to shows they haven't seen yet.

How does it do that? It figures out hidden patterns (called **factors**) — things like *"this user loves sci-fi"* and *"this show is very sci-fi"*. If a user's taste lines up with a show's qualities, ALS predicts a high rating. It works by going back and forth:
1. First it locks the user patterns and figures out the show patterns
2. Then it locks the show patterns and figures out the user patterns
3. It repeats this over and over until the predictions stop improving

That back-and-forth is the **"Alternating"** part of Alternating Least Squares!

### Key Hyperparameters
| Parameter | What It Controls | Range We'll Try |
|-----------|-----------------|----------------|
| `rank` | How many hidden taste/quality patterns to look for (more = more nuanced, but slower) | 10 – 100 |
| `maxIter` | How many back-and-forth rounds to run | 10 – 15 |
| `regParam` | How much we prevent the model from overthinking (overfitting) the training data | 0.01 – 0.5 |

### MLflow Integration
We use MLflow to:
- **Track** each combination of parameters and how well it performed (RMSE)
- **Compare** runs side-by-side in the MLflow UI
- **Register** the winning model to Unity Catalog so it can be deployed to production

### Step 3.1 — Prepare Train/Test Split for ALS

1. Run the cell directly below (**Shift + Enter**)
2. **What you should see:** Counts for the ALS training set (~80%) and validation set (~20%), plus a rating distribution table showing how many interactions exist at each rating level
3. **What it does:** Loads the interaction matrix we saved in Section 2 and splits it 80/20 for hyperparameter tuning

In [0]:
# --- Prepare ALS Input ---
# Load the interaction matrix we saved earlier
df_als_input = spark.table(f"{SCHEMA}.training_interactions")

# Split into train/validation (80/20) for hyperparameter tuning
# Using random split since our point-in-time split is already handled
(als_train, als_val) = df_als_input.randomSplit([0.8, 0.2], seed=42)

print(f"ALS Training set:   {als_train.count():,} interactions")
print(f"ALS Validation set: {als_val.count():,} interactions")

# Preview the interaction data
print("Rating Distribution:")
display(
    df_als_input
    .withColumn("rating_bucket", F.round("rating", 0))
    .groupBy("rating_bucket")
    .count()
    .orderBy("rating_bucket")
)

### Step 3.2 — 🧪 Exercise 3: Configure ALS and Run Hyperparameter Tuning

> **This is a coding exercise.** Replace each `<FILL_IN>` placeholder with the correct value.

1. Read through the code in the cell below and replace each `<FILL_IN>` with the correct value
2. Run the cell (**Shift + Enter**)
3. **What you should see:** Progress output for 18 combinations (3 ranks × 2 maxIter × 3 regParam), each showing an RMSE score. The best model is marked with a ⭐. This cell takes **3–5 minutes**
4. **What it does:** Trains 18 ALS models with different hyperparameter combinations, logs each to MLflow, and keeps track of the best one

**Hints:**
* ALS needs to know which column contains the **user identifiers** — look at the interaction matrix you created in Exercise 2
* ALS needs to know which column contains the **item identifiers** — what's the content column name?
* ALS needs to know which column contains the **ratings** to learn from — what did you name the rating column in Exercise 2?

In [0]:
# --- Hyperparameter Tuning with MLflow ---
# We'll run a grid search, logging each combination to MLflow

from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

# Set up MLflow experiment
mlflow.set_experiment(EXPERIMENT_NAME)

# Define hyperparameter grid
param_grid = {
    "rank": [10, 50, 100],
    "maxIter": [10, 15],
    "regParam": [0.01, 0.1, 0.5]
}

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

best_rmse = float("inf")
best_model = None
best_params = {}
run_count = 0
total_runs = len(param_grid["rank"]) * len(param_grid["maxIter"]) * len(param_grid["regParam"])

print(f"Starting hyperparameter search: {total_runs} combinations")
print("=" * 60)

for rank in param_grid["rank"]:
    for maxIter in param_grid["maxIter"]:
        for regParam in param_grid["regParam"]:
            run_count += 1
            run_name = f"ALS_r{rank}_i{maxIter}_reg{regParam}"
            
            with mlflow.start_run(run_name=run_name) as run:
                # Log parameters
                mlflow.log_param("rank", rank)
                mlflow.log_param("maxIter", maxIter)
                mlflow.log_param("regParam", regParam)
                mlflow.log_param("algorithm", "ALS")
                mlflow.log_param("feedback_type", "explicit")
                
                # =============================================================
                # EXERCISE 3: Configure the ALS model
                # Replace each <FILL_IN> with the correct column name.
                # =============================================================
                als = ALS(
                    rank=rank,
                    maxIter=maxIter,
                    regParam=regParam,
                    userCol=<FILL_IN>,           # Column name for user IDs (as a string)
                    itemCol=<FILL_IN>,           # Column name for content/item IDs (as a string)
                    ratingCol=<FILL_IN>,         # Column name for ratings (as a string)
                    coldStartStrategy="drop",    # Drop NaN predictions for new users/items
                    nonnegative=True             # Ratings are positive
                )
                
                model = als.fit(als_train)
                
                # Evaluate on validation set
                predictions = model.transform(als_val)
                rmse = evaluator.evaluate(predictions)
                
                # Log metrics
                mlflow.log_metric("rmse", rmse)
                mlflow.log_metric("num_user_factors", model.userFactors.count())
                mlflow.log_metric("num_item_factors", model.itemFactors.count())
                
                # Track best model
                if rmse < best_rmse:
                    best_rmse = rmse
                    best_model = model
                    best_params = {"rank": rank, "maxIter": maxIter, "regParam": regParam}
                    best_run_id = run.info.run_id
                
                status = " ⭐ BEST" if rmse == best_rmse else ""
                print(f"  [{run_count}/{total_runs}] {run_name}: RMSE={rmse:.4f}{status}")

print("\n" + "=" * 60)
print(f"Best Model: RMSE={best_rmse:.4f}")
print(f" Parameters: {best_params}")
print(f"Run ID: {best_run_id}")

### Step 3.3 — Evaluate the Best Model and Generate Recommendations

1. Run the cell directly below (**Shift + Enter**)
2. **What you should see:** The best model's RMSE and parameters, sample recommendations for User 1 (10 items with title, genre, and predicted rating), and a confirmation that recommendations were saved
3. **This cell takes 2–3 minutes** because it generates predictions for all user×item pairs and ranks them
4. **What it does:** Uses the best ALS model to predict ratings for every user-item combination, ranks the top 10 per user, and saves them as a Delta table

In [0]:
# --- Evaluate the Best Model ---
print(f"Best ALS Model Performance")
print(f"RMSE: {best_rmse:.4f}")
print(f"Params: rank={best_params['rank']}, maxIter={best_params['maxIter']}, regParam={best_params['regParam']}")

# Generate top-10 recommendations for all users
# Using transform() + window ranking instead of recommendForAllUsers()
# to avoid higher-order functions unsupported on UC serverless
df_all_users = df_als_input.select("user_id").distinct()
df_all_content = df_als_input.select("content_id").distinct()
df_all_pairs = df_all_users.crossJoin(df_all_content)

predictions = best_model.transform(df_all_pairs).filter(F.col("prediction").isNotNull())

w = Window.partitionBy("user_id").orderBy(F.desc("prediction"))
df_recs_exploded = (
    predictions
    .withColumn("rank", F.row_number().over(w))
    .filter(F.col("rank") <= 10)
    .select(
        "user_id",
        "rank",
        "content_id",
        F.round("prediction", 2).alias("predicted_rating")
    )
    .join(df_content.select("content_id", "title", "genre"), "content_id")
)

print("Sample Recommendations (User 1):")
display(
    df_recs_exploded
    .filter(F.col("user_id") == 1)
    .orderBy("rank")
)

# Save recommendations for later use
df_recs_exploded.write.mode("overwrite").saveAsTable(f"{SCHEMA}.user_recommendations")
print(f"\n\u2705 Recommendations saved to {SCHEMA}.user_recommendations")

### Step 3.4 — Recommendation Quality Analysis

1. Run the cell directly below (**Shift + Enter**)
2. **What you should see:** A genre preference match rate (what % of recommendations match users' preferred genres), plus a distribution table showing how many users fall into each match rate bucket
3. **Key insight:** A random baseline would match ~30% (3 of 10 genres). If the model exceeds this, it's learning meaningful preferences

In [0]:
# --- Recommendation Quality Check ---
# Verify our recommendations make sense by comparing them
# to users' known genre preferences

# For each user, check what % of recommendations match their preferred genres
df_quality = (
    df_recs_exploded
    .join(df_users.select("user_id", "preferred_genres"), "user_id")
    .withColumn("genre_match", 
        F.when(
            F.col("preferred_genres").contains(F.col("genre")), 1
        ).otherwise(0)
    )
    .groupBy("user_id")
    .agg(
        F.avg("genre_match").alias("preference_match_rate"),
        F.avg("predicted_rating").alias("avg_predicted_score"),
        F.count("*").alias("num_recs")
    )
)

overall_match = df_quality.agg(F.avg("preference_match_rate")).first()[0]

print(f"🎯 Recommendation Quality Analysis")
print("=" * 60)
print(f"  Genre Preference Match Rate: {overall_match:.1%}")
print(f"  (% of recommended content matching user's preferred genres)")
print(f"  Random baseline would be ~30% (3 of 10 genres)")
print(f"  Our model achieves {overall_match:.1%} → {'✅ Above random!' if overall_match > 0.30 else '⚠️ Needs improvement'}")

print("\n📊 Match Rate Distribution:")
display(
    df_quality
    .withColumn("match_bucket", 
        F.when(F.col("preference_match_rate") >= 0.8, "80-100%")
        .when(F.col("preference_match_rate") >= 0.6, "60-80%")
        .when(F.col("preference_match_rate") >= 0.4, "40-60%")
        .when(F.col("preference_match_rate") >= 0.2, "20-40%")
        .otherwise("0-20%")
    )
    .groupBy("match_bucket")
    .agg(F.count("*").alias("num_users"))
    .orderBy("match_bucket")
)

### Step 3.5 — 🧪 Exercise 4: Register the Best Model to Unity Catalog

> **This is a coding exercise.** Replace each `<FILL_IN>` placeholder with the correct value.

1. Read through the code in the cell below and replace each `<FILL_IN>` with the correct value
2. Run the cell (**Shift + Enter**)
3. **What you should see:** Confirmation that the model was registered to Unity Catalog with a run ID, and that the "Champion" alias was set to the latest version
4. **What it does:** Logs the best ALS model to MLflow, registers it in Unity Catalog for versioning and lineage, and sets an alias so downstream systems can reference the production model

**Hints:**
* `registered_model_name` should point to the Unity Catalog model path — which variable holds that path? (Look at the Configuration cell in Section 1)
* `artifact_path` is the name under which the model artifacts are stored in the MLflow run — use a descriptive string like `"als_model"`
* For the alias, we want to set the name `"Champion"` on the latest version — what string literal marks the production model?

In [0]:
# The ALS model is logged to UC for versioning, lineage, and reproducibility.
# Serving is handled by Feature Serving (Part 3), not by a model serving endpoint.
from mlflow.models import infer_signature

mlflow.set_registry_uri("databricks-uc")

# UC Volume for temporary Spark model storage (required on serverless)
VOLUME_PATH = f"/Volumes/content_reco_lab/content_reco_demo/ml_artifacts"
spark.sql(f"CREATE VOLUME IF NOT EXISTS {SCHEMA}.ml_artifacts")

sample_input = als_train.select("user_id", "content_id").limit(5).toPandas()

# =============================================================================
# EXERCISE 4: Register the model to Unity Catalog
# Replace each <FILL_IN> with the correct value.
# =============================================================================

with mlflow.start_run(run_name="best_als_model_registration") as run:
    mlflow.log_params(best_params)
    mlflow.log_metric("rmse", best_rmse)

    mlflow.spark.log_model(
        spark_model=best_model,
        artifact_path=<FILL_IN>,                   # What name to store model artifacts under? (a string)
        registered_model_name=<FILL_IN>,           # Which variable holds the UC model path?
        input_example=sample_input,
        dfs_tmpdir=f"{VOLUME_PATH}/tmp",
    )

    print(f"\n\u2705 Model registered to Unity Catalog: {MODEL_NAME}")
    print(f"   Run ID: {run.info.run_id}")

from mlflow import MlflowClient
client = MlflowClient()
latest_versions = client.search_model_versions(f"name='{MODEL_NAME}'")
if latest_versions:
    latest_version = max(int(v.version) for v in latest_versions)
    client.set_registered_model_alias(MODEL_NAME, <FILL_IN>, str(latest_version))  # What alias marks the production model? (a string)
    print(f"   Alias 'Champion' set to version {latest_version}")

> ✅ **Checkpoint:** You have trained an ALS model with hyperparameter tuning, evaluated recommendation quality, and registered the best model to Unity Catalog with a "Champion" alias. **Scroll down to continue to Section 4.**

---
# Section 4: Real-Time Serving with Feature Serving

**Goal:** Precompute recommendation tables, sync them to a low-latency online store, and expose a REST API that serves personalized picks in under 10 ms.

---

### Why Not a Model Serving Endpoint?

Our ALS model is built with PySpark, which needs a full **JVM** to load. Model Serving containers are lightweight, CPU-constrained environments — not designed to spin up Spark. Instead, we use a simpler, faster approach.

### The Key Insight: Recommendations Are Lookups, Not Computations

| Widget | What the app needs | How often it changes |
|--------|-------------------|---------------------|
| **"Top 10 For You"** | Given a `user_id`, return their best content | Once per retrain (nightly) |
| **"Because You Watched X"** | Given a `content_id`, return similar items | Once per retrain (nightly) |

Both are **static lookups** — the answers only change when the model is retrained. So instead of running a model on every request, we:
1. **Precompute** all the answers in batch
2. **Store** them in wide-format Delta tables (1 row per key)
3. **Sync** them to a low-latency **Online Store** (Lakebase)
4. **Serve** them via a **Feature Serving endpoint** as fast key-value lookups

### Architecture
```
  ┌──────────────┐     ┌───────────────────┐     ┌───────────────┐     ┌──────────────┐
  │  ALS Model    │───▶│  Delta Tables      │───▶│  Online Store  │───▶│  REST API     │
  │  (Spark)      │     │  (wide format)     │     │  (Lakebase)    │     │  (Feature    │
  │              │     │  1 row per key     │     │  auto-sync     │     │   Serving)   │
  └──────────────┘     └───────────────────┘     └───────────────┘     └──────────────┘
       Batch              This Section             Lakebase            <10ms lookups
     (nightly)         PK + CDF enabled          publish_table
```

### Two Precomputed Tables

**`serving_user_recs`** — One row per `user_id` with 10 recommendation slots (`rec_1_content_id`, `rec_1_title`, `rec_1_score`, ... `rec_10_*`)

**`serving_item_similarities`** — One row per `content_id` with 10 similar-item slots (`sim_1_content_id`, `sim_1_title`, `sim_1_score`, ... `sim_10_*`)

Follow the steps below in order.

### Step 4.1 — Build the Item Similarity Table

1. Run the cell directly below (**Shift + Enter**)
2. **What you should see:** Progress output for computing cosine similarities from ALS item factors, followed by a confirmation message and a preview of the wide-format similarity table
3. **This cell takes 1–2 minutes** to compute pairwise similarities across all items
4. **What it does:** Extracts item factor vectors from the ALS model, computes cosine similarity between all pairs, and creates a Delta table where each row has one `content_id` and its top 10 most similar items

In [0]:
# --- Prepare Feature Serving Tables ---
# Instead of deploying a model serving endpoint, we precompute recommendations
# and serve them as low-latency key-value lookups via Feature Serving.
#
# Two tables (wide format: 1 row per key):
#   1. serving_user_recs:          user_id    → "Top 10 For You"
#   2. serving_item_similarities:  content_id → "Because You Watched X"

import numpy as np
import pandas as pd

# --- Step 1: Compute Item-to-Item Cosine Similarity ---
print("Step 1: Computing item-to-item similarities from ALS factors...")
item_factors_pdf = best_model.itemFactors.select("id", "features").toPandas()
item_ids = item_factors_pdf["id"].values.astype(int)
item_matrix = np.stack(item_factors_pdf["features"].values).astype(np.float32)

# Cosine similarity
norms = np.linalg.norm(item_matrix, axis=1, keepdims=True)
item_normed = item_matrix / np.maximum(norms, 1e-10)
sim_matrix = item_normed @ item_normed.T

# Content metadata lookup
content_pdf = df_content.select("content_id", "title", "genre").toPandas()
id_to_meta = {
    int(r["content_id"]): (r["title"], r["genre"])
    for _, r in content_pdf.iterrows()
}

# Build wide-format table (1 row per content_id, top-10 similar items)
TOP_N = 10
sim_rows = []
for i, cid in enumerate(item_ids):
    sims = sim_matrix[i].copy()
    sims[i] = -1  # exclude self
    top_idx = np.argsort(sims)[::-1][:TOP_N]

    row = {"content_id": int(cid)}
    for rank, idx in enumerate(top_idx, 1):
        sid = int(item_ids[idx])
        title, genre = id_to_meta.get(sid, ("Unknown", "Unknown"))
        row[f"sim_{rank}_content_id"] = sid
        row[f"sim_{rank}_title"] = title
        row[f"sim_{rank}_genre"] = genre
        row[f"sim_{rank}_score"] = round(float(sims[idx]), 4)
    sim_rows.append(row)

df_item_sim = spark.createDataFrame(pd.DataFrame(sim_rows))
df_item_sim.write.mode("overwrite").saveAsTable(f"{SCHEMA}.serving_item_similarities")
spark.sql(f"ALTER TABLE {SCHEMA}.serving_item_similarities ALTER COLUMN content_id SET NOT NULL")
spark.sql(f"ALTER TABLE {SCHEMA}.serving_item_similarities DROP CONSTRAINT IF EXISTS item_sim_pk")
spark.sql(f"ALTER TABLE {SCHEMA}.serving_item_similarities ADD CONSTRAINT item_sim_pk PRIMARY KEY (content_id)")
spark.sql(f"ALTER TABLE {SCHEMA}.serving_item_similarities SET TBLPROPERTIES (delta.enableChangeDataFeed = 'true')")
print(f"  \u2705 {SCHEMA}.serving_item_similarities ({len(sim_rows)} items \u00d7 top-{TOP_N})")

display(df_item_sim)

### Step 4.2 — Build the User Recommendations Table

1. Run the cell directly below (**Shift + Enter**)
2. **What you should see:** A confirmation message and a preview of the wide-format user recommendations table
3. **What it does:** Pivots the user recommendations (generated in Section 3) from long format to wide format — one row per user with columns for each recommendation slot

In [0]:
# --- Step 2: Pivot User Recommendations to Wide Format ---
print("\nStep 2: Pivoting user recommendations to wide format...")
recs_pdf = spark.table(f"{SCHEMA}.user_recommendations").toPandas()
rec_rows = []
for uid, group in recs_pdf.groupby("user_id"):
    group = group.sort_values("rank")
    row = {"user_id": int(uid)}
    for _, r in group.head(TOP_N).iterrows():
        rank = int(r["rank"])
        row[f"rec_{rank}_content_id"] = int(r["content_id"])
        row[f"rec_{rank}_title"] = r["title"]
        row[f"rec_{rank}_genre"] = r["genre"]
        row[f"rec_{rank}_score"] = round(float(r["predicted_rating"]), 4)
    rec_rows.append(row)

df_user_recs = spark.createDataFrame(pd.DataFrame(rec_rows))
df_user_recs.write.mode("overwrite").saveAsTable(f"{SCHEMA}.serving_user_recs")
spark.sql(f"ALTER TABLE {SCHEMA}.serving_user_recs ALTER COLUMN user_id SET NOT NULL")
spark.sql(f"ALTER TABLE {SCHEMA}.serving_user_recs DROP CONSTRAINT IF EXISTS user_recs_pk")
spark.sql(f"ALTER TABLE {SCHEMA}.serving_user_recs ADD CONSTRAINT user_recs_pk PRIMARY KEY (user_id)")
spark.sql(f"ALTER TABLE {SCHEMA}.serving_user_recs SET TBLPROPERTIES (delta.enableChangeDataFeed = 'true')")
print(f"  \u2705 {SCHEMA}.serving_user_recs ({len(rec_rows)} users \u00d7 top-{TOP_N})")

# --- Preview ---
print("\n\U0001f4cb Sample: User recommendations (wide format):")
display(df_user_recs)

   
### 🗄️ What Is an Online Store? What Is Lakebase?

**Delta tables are built for analytics** — scanning millions of rows, running aggregations, training models. They're *not* designed for single-row, sub-10ms lookups that a production app needs when a user opens their homepage.

That's where the **Online Feature Store** comes in. It's a separate, high-performance serving layer purpose-built for **point lookups**: given a key (like `user_id`), return that one row instantly.

**Lakebase** is the engine that powers it. Under the hood, it's a fully managed **Postgres database** integrated into the Databricks platform. When you "publish" a Delta table to an online store, Databricks syncs the data into Lakebase so it can be read at operational speed.

| | Delta Table (Offline) | Lakebase (Online) |
|---|---|---|
| **Optimized for** | Analytics, batch, ML training | Single-row lookups, OLTP |
| **Latency** | Seconds to minutes | Sub-10 ms |
| **Access pattern** | Scan millions of rows | Fetch one row by primary key |
| **Kept in sync via** | — | Change Data Feed (CDF) from the source Delta table |

> **Think of it this way:** Delta is your warehouse. Lakebase is the vending machine stocked from that warehouse — fast to access, limited to what's been loaded, and automatically restocked when the source changes.

### Step 4.3 — Create the Online Store and Publish Tables

1. Run the cell directly below (**Shift + Enter**)
2. **What you should see:** Three steps completing:
   - Step 1: Online store created (or already exists)
   - Step 2: Both tables published to the online store
3. **This cell takes 2–3 minutes** as it creates the Lakebase online store and syncs the Delta tables
4. **What it does:** Creates a Lakebase online store, then publishes both serving tables to it so they can be accessed at sub-10ms latency

In [0]:
# --- Feature Serving Endpoint ---
# Serve precomputed recommendations via low-latency key-value lookup.
# No custom model, no pyfunc class — just table lookups!
#
# Architecture:
#   Delta Table → Online Store (Lakebase) → Feature Spec → REST Endpoint

# Re-define constants (Python was restarted for package install)
SCHEMA = "content_reco_lab.content_reco_demo"
ENDPOINT_NAME = "content-recommender-endpoint"

from databricks.feature_engineering import FeatureLookup, FeatureEngineeringClient
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

fe = FeatureEngineeringClient()
w = WorkspaceClient()

ONLINE_STORE_NAME = "content-reco-online-store"
FEATURE_SPEC_NAME = f"{SCHEMA}.content_recommender_spec"

# --- Step 1: Create Online Store ---
print("Step 1: Creating online store...")
try:
    fe.create_online_store(name=ONLINE_STORE_NAME, capacity="CU_1")
    print(f"  \u2705 Created: {ONLINE_STORE_NAME}")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"  \u2139\ufe0f  Already exists: {ONLINE_STORE_NAME}")
    else:
        raise

# --- Step 2: Publish Feature Tables to Online Store ---
print("\nStep 2: Publishing tables to online store...")
online_store = fe.get_online_store(name=ONLINE_STORE_NAME)

publish_configs = {
    "serving_user_recs": "ot_serving_user_recs",
    "serving_item_similarities": "ot_serving_item_similarities",
}

for source_table, online_table in publish_configs.items():
    print(f"  Publishing {source_table}...")
    try:
        fe.publish_table(
            online_store=online_store,
            source_table_name=f"{SCHEMA}.{source_table}",
            online_table_name=f"{SCHEMA}.{online_table}",
        )
        print(f"  \u2705 Published: {source_table} \u2192 {online_table}")
    except Exception as e:
        if "already exists" in str(e).lower():
            print(f"  \u2139\ufe0f  Already published: {online_table}")
        else:
            raise

   
### 📋 What Is a Feature Spec?

A **Feature Spec** is a Unity Catalog object that defines *which tables to look up and which keys to use* when serving data. Think of it as a **wiring diagram** for the Feature Serving endpoint.

In our case, the spec says:

| Lookup Key | Source Table | What It Returns |
|---|---|---|
| `user_id` | `serving_user_recs` | That user's top-10 personalized recommendations |
| `content_id` | `serving_item_similarities` | The top-10 items similar to that content |

When a request hits the endpoint with `{user_id: 42, content_id: 147}`, the Feature Spec tells the serving layer: *"Look up user 42 in the first table and content 147 in the second table, then combine and return the results."*

The spec itself holds no data — it's purely metadata that maps keys → tables → columns.

### Step 4.4 — 🧪 Exercise 5: Create the Feature Spec

> **This is a coding exercise.** Replace each `<FILL_IN>` placeholder with the correct value.

1. Read through the code in the cell below and replace each `<FILL_IN>` with the correct value
2. Run the cell (**Shift + Enter**)
3. **What you should see:** A confirmation that the feature spec was created (or already exists)
4. **What it does:** Creates a Feature Spec in Unity Catalog that maps lookup keys to the online tables

**Hints:**
* The first `FeatureLookup` is for personalized recommendations — which table holds the user recs? (Use the fully qualified name with the `SCHEMA` variable)
* The `lookup_key` for user recs is the column that uniquely identifies a user
* The second `FeatureLookup` is for similar items — which table holds the item similarities?
* The `lookup_key` for item similarities is the column that uniquely identifies a content item

In [0]:
# --- Step 3: Create Feature Spec ---
# =============================================================================
# EXERCISE 5: Define the Feature Spec lookups
# The Feature Spec maps lookup keys to source tables.
# Replace each <FILL_IN> with the correct value.
# =============================================================================

print("\nStep 3: Creating feature spec...")
features = [
    FeatureLookup(
        table_name=f"{SCHEMA}.<FILL_IN>",     # Which table has the user recommendations?
        lookup_key=<FILL_IN>,                  # Which column is the primary key for user recs? (a string)
    ),
    FeatureLookup(
        table_name=f"{SCHEMA}.<FILL_IN>",     # Which table has the item similarities?
        lookup_key=<FILL_IN>,                  # Which column is the primary key for item similarities? (a string)
    ),
]

try:
    fe.create_feature_spec(name=FEATURE_SPEC_NAME, features=features)
    print(f"  \u2705 Created: {FEATURE_SPEC_NAME}")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"  \u2139\ufe0f  Already exists: {FEATURE_SPEC_NAME}")
    else:
        raise

### Step 4.5 — Create the Serving Endpoint

1. Run the cell directly below (**Shift + Enter**)
2. **What you should see:** Either an update to an existing endpoint or creation of a new one, followed by a "🎉 Feature Serving is live!" message
3. **This cell takes 5–10 minutes** if creating a new endpoint (it provisions infrastructure). If the endpoint already exists, it updates in about 1 minute
4. **What it does:** Creates (or updates) a managed REST API endpoint backed by the Feature Spec. The endpoint reads from Lakebase for sub-10ms responses

In [0]:
# --- Step 4: Create/Update Serving Endpoint ---
print(f"\nStep 4: Setting up endpoint '{ENDPOINT_NAME}'...")
served_entity = ServedEntityInput(
    entity_name=FEATURE_SPEC_NAME,
    scale_to_zero_enabled=True,
    workload_size="Small",
)

try:
    existing = w.serving_endpoints.get(ENDPOINT_NAME)
    print(f"  Endpoint exists (state: {existing.state.ready}). Updating...")
    w.serving_endpoints.update_config_and_wait(
        name=ENDPOINT_NAME,
        served_entities=[served_entity]
    )
    print(f"  \u2705 Endpoint updated: {ENDPOINT_NAME}")
except Exception as e:
    if "RESOURCE_DOES_NOT_EXIST" in str(e) or "does not exist" in str(e).lower():
        print(f"  Creating endpoint (5-10 minutes)...")
        w.serving_endpoints.create_and_wait(
            name=ENDPOINT_NAME,
            config=EndpointCoreConfigInput(
                name=ENDPOINT_NAME,
                served_entities=[served_entity]
            )
        )
        print(f"  \u2705 Endpoint READY: {ENDPOINT_NAME}")
    else:
        raise

print(f"\n\U0001f389 Feature Serving is live!")
print(f"   Endpoint: {ENDPOINT_NAME}")
print(f"   Send {{user_id, content_id}} \u2192 get personalized recs + similar items")

### Step 4.6 — 🧪 Exercise 6: Test the Serving Endpoint

> **This is a coding exercise.** Replace each `<FILL_IN>` placeholder with the correct value.

1. Read through the code in the cell below and replace each `<FILL_IN>` with the correct value
2. Run the cell (**Shift + Enter**)
3. **What you should see:** A formatted display showing:
   - "Top 10 For You" — personalized recommendations for the user
   - "Because You Watched Content X" — items similar to the specified content
4. **What it does:** Sends a request to the Feature Serving endpoint simulating a user who just finished watching a show

**Hints:**
* The `endpoint` parameter should reference the endpoint name variable defined earlier in this section
* The request payload uses `"dataframe_records"` format with a list of dictionaries
* The keys in the dictionary must match the lookup keys in the Feature Spec: one for the user and one for the content item
* Use any valid user ID (1–500) and content ID (1–200)

In [0]:
# --- Test Feature Serving Endpoint ---
# Scenario: A user just finished watching a show.
# The app calls the endpoint with {user_id, content_id} and gets:
#   • "Top 10 For You" — personalized recommendations
#   • "Because You Watched" — items similar to what they watched

import mlflow.deployments

client = mlflow.deployments.get_deploy_client("databricks")

# =============================================================================
# EXERCISE 6: Call the serving endpoint
# Replace each <FILL_IN> with the correct value.
# =============================================================================

response = client.predict(
    endpoint=<FILL_IN>,             # Which variable holds the endpoint name?
    inputs={
        "dataframe_records": [
            {<FILL_IN>: 42, <FILL_IN>: 147}   # What are the two lookup key names? (as strings)
        ]
    },
)

result = response["outputs"][0]

print("\U0001f3ac User 42 just watched content 147")
print("=" * 60)

# Display personalized recommendations
print("\n\U0001f4cb Top 10 For You (personalized):")
for i in range(1, 11):
    title = result.get(f"rec_{i}_title", "\u2014")
    genre = result.get(f"rec_{i}_genre", "\u2014")
    score = result.get(f"rec_{i}_score", 0)
    if title != "\u2014":
        print(f"  {i:>2}. [{genre}] {title} (score: {score})")

# Display similar items
print(f"\n\U0001f517 Because You Watched Content 147:")
for i in range(1, 11):
    title = result.get(f"sim_{i}_title", "\u2014")
    genre = result.get(f"sim_{i}_genre", "\u2014")
    score = result.get(f"sim_{i}_score", 0)
    if title != "\u2014":
        print(f"  {i:>2}. [{genre}] {title} (similarity: {score})")

print(f"\n\u2705 Feature Serving: <10ms latency, no model inference \u2014 just lookups!")

   
### In production, a nightly Databricks Job refreshes recommendations:
  1. Ingest new viewing data from streaming pipeline
  2. Retrain ALS model (or incrementally update factors)
  3. Regenerate user_recommendations
  4. Regenerate serving tables
  5. Re-publish to Online Store → Feature Serving auto-updates

The Feature Serving endpoint always serves the latest published data.
No model redeployment needed — just refresh the underlying tables.

> ✅ **Checkpoint:** You have built two precomputed serving tables, synced them to Lakebase, created a Feature Spec, deployed a Feature Serving endpoint, and tested it with a live API call. The endpoint returns personalized recommendations and similar items in under 10 ms. **Scroll down to continue to Section 5.**

---
# Section 5: Cleanup — Remove All Demo Resources

**Goal:** Tear down everything created by this lab so your workspace is restored to its original state.

### Instructions

1. Run the cleanup cell below (**Shift + Enter**)
2. The cell is idempotent — safe to re-run if anything fails partway through
3. **Warning:** This is irreversible. It removes the catalog, schema, all tables, the serving endpoint, online store, MLflow experiment, and registered model

---

In [0]:
# --- OPTIONAL: Full Cleanup ---
# Removes ALL resources created by this notebook AND 00_Setup_Data_Generation.
# WARNING: This is irreversible. Uncomment the line at the bottom to execute.

def cleanup_lab():
    from databricks.sdk import WorkspaceClient
    from databricks.feature_engineering import FeatureEngineeringClient
    from mlflow import MlflowClient
    import mlflow

    SCHEMA = "content_reco_lab.content_reco_demo"
    CATALOG = "content_reco_lab"
    MODEL_NAME = f"{SCHEMA}.content_recommender_als"
    ENDPOINT_NAME = "content-recommender-endpoint"
    ONLINE_STORE_NAME = "content-reco-online-store"
    FEATURE_SPEC_NAME = f"{SCHEMA}.content_recommender_spec"
    EXPERIMENT_NAME = "/Users/{}/content_reco_experiment".format(
        spark.sql("SELECT current_user()").first()[0]
    )

    # Online table names used by publish_table (must match the publish cell)
    ONLINE_TABLE_NAMES = [
        f"{SCHEMA}.ot_serving_user_recs",
        f"{SCHEMA}.ot_serving_item_similarities",
    ]

    w = WorkspaceClient()
    fe = FeatureEngineeringClient()
    mlflow.set_registry_uri("databricks-uc")
    ml_client = MlflowClient()

    errors = []

    def safe(label, fn):
        try:
            fn()
            print(f"  \u2705 {label}")
        except Exception as e:
            if "does not exist" in str(e).lower() or "not found" in str(e).lower() or "resource_does_not_exist" in str(e).upper():
                print(f"  \u2139\ufe0f  {label} (already gone)")
            else:
                errors.append((label, e))
                print(f"  \u26a0\ufe0f  {label}: {e}")

    print("\U0001f9f9 Starting full lab cleanup...")
    print("=" * 60)

    # 1. Delete serving endpoint
    print("\n[1/8] Serving Endpoint")
    safe(f"Delete endpoint '{ENDPOINT_NAME}'",
         lambda: w.serving_endpoints.delete(ENDPOINT_NAME))

    # 2. Delete feature spec
    print("\n[2/8] Feature Spec")
    safe(f"Delete feature spec '{FEATURE_SPEC_NAME}'",
         lambda: fe.delete_feature_spec(name=FEATURE_SPEC_NAME))

    # 3. Delete the online store
    print("\n[3/8] Online Store")
    try:
        online_store = fe.get_online_store(name=ONLINE_STORE_NAME)
        if online_store:
            safe(f"Delete online store '{ONLINE_STORE_NAME}'",
                 lambda: fe.delete_online_store(name=ONLINE_STORE_NAME))
        else:
            print(f"  \u2139\ufe0f  Online store '{ONLINE_STORE_NAME}' (already gone)")
    except Exception:
        print(f"  \u2139\ufe0f  Online store '{ONLINE_STORE_NAME}' (already gone)")

    # 4. Delete DLT sync pipelines created by publish_table
    print("\n[4/8] Sync Pipelines (created by publish_table)")
    try:
        all_pipelines = list(w.pipelines.list_pipelines())
        sync_keywords = ONLINE_TABLE_NAMES + [ONLINE_STORE_NAME, "ot_serving_user_recs", "ot_serving_item_similarities"]
        matched = [
            p for p in all_pipelines
            if p.name and any(kw in p.name for kw in sync_keywords)
        ]
        if matched:
            for p in matched:
                safe(f"Delete sync pipeline '{p.name}' ({p.pipeline_id})",
                     lambda pid=p.pipeline_id: w.pipelines.delete(pipeline_id=pid))
        else:
            print("  \u2139\ufe0f  No matching sync pipelines found")
    except Exception as e:
        errors.append(("List/delete sync pipelines", e))
        print(f"  \u26a0\ufe0f  Error listing pipelines: {e}")

    # 5. Delete registered model (all versions)
    print("\n[5/8] MLflow Model")
    def delete_model():
        versions = ml_client.search_model_versions(f"name='{MODEL_NAME}'")
        for v in versions:
            ml_client.delete_model_version(MODEL_NAME, v.version)
        ml_client.delete_registered_model(MODEL_NAME)
    safe(f"Delete model '{MODEL_NAME}'", delete_model)

    # 6. Delete MLflow experiment and all runs
    print("\n[6/8] MLflow Experiment")
    def delete_experiment():
        exp = ml_client.get_experiment_by_name(EXPERIMENT_NAME)
        if exp:
            ml_client.delete_experiment(exp.experiment_id)
        else:
            raise Exception("not found")
    safe(f"Delete experiment '{EXPERIMENT_NAME}'", delete_experiment)

    # 7. Drop schema CASCADE
    print("\n[7/8] Schema (all tables + volumes)")
    safe(f"Drop schema '{SCHEMA}' CASCADE",
         lambda: spark.sql(f"DROP SCHEMA IF EXISTS {SCHEMA} CASCADE"))

    # 8. Drop catalog
    print("\n[8/8] Catalog")
    safe(f"Drop catalog '{CATALOG}'",
         lambda: spark.sql(f"DROP CATALOG IF EXISTS {CATALOG} CASCADE"))

    # Summary
    print("\n" + "=" * 60)
    if errors:
        print(f"\u26a0\ufe0f  Cleanup finished with {len(errors)} issue(s):")
        for label, e in errors:
            print(f"   - {label}: {e}")
    else:
        print("\u2705 All lab resources cleaned up successfully!")

# \u26a0\ufe0f Uncomment the line below to run the full cleanup:
# cleanup_lab()

## Lab Complete!
Congratulations! You've successfully completed the Content Recommendation Engine lab.

---

### Key Takeaways

| Concept | What You Learned |
|---|---|
| **Feature Engineering** | Transforming raw viewing events into user-level and content-level ML features using PySpark aggregations and window functions |
| **Point-in-Time Correctness** | Splitting data at a training cutoff to prevent data leakage and ensure features reflect only past information |
| **ALS Collaborative Filtering** | Training an Alternating Least Squares model to predict user-item ratings by learning latent factor patterns |
| **MLflow Tracking** | Logging hyperparameter combinations, metrics, and model artifacts for reproducible experiment management |
| **Unity Catalog Model Registry** | Registering trained models with versioning, lineage, and alias-based promotion ("Champion") |
| **Precomputed Serving Tables** | Pivoting recommendations into wide-format Delta tables with primary keys and Change Data Feed enabled |
| **Online Store (Lakebase)** | Syncing Delta tables to a managed Postgres-backed store for sub-10ms key-value lookups |
| **Feature Specs** | Defining declarative lookup mappings (key → table → columns) in Unity Catalog |
| **Feature Serving Endpoints** | Deploying a managed REST API that combines multiple table lookups into a single response with auto-scaling |
| **Cosine Similarity** | Computing item-to-item similarity from ALS factor vectors for "Because You Watched" recommendations |

---

### Production Next Steps
1. **Schedule a nightly Databricks Job** to retrain the model, regenerate serving tables, and re-publish to the online store
2. **Add A/B testing** by deploying multiple model versions behind the same endpoint and tracking click-through rates
3. **Incorporate implicit feedback** (watch completion %, time-of-day) alongside explicit ratings for richer signals
4. **Add real-time features** (current session clicks, trending content) via streaming feature pipelines
5. **Monitor model drift** by comparing recommendation distributions over time using MLflow and Lakehouse Monitoring

---

# Answer Key

> **⚠️ Spoiler alert!** The complete solutions for all 6 exercises are below. Try to solve each exercise on your own first — you'll learn much more that way. Use these answers only when you're truly stuck.

## Exercise 1 — Engineer User-Level Features

Copy the complete cell below into the Exercise 1 cell:

```python
# --- User-Level Feature Engineering ---
# These features capture each user's viewing behavior and preferences.
# In production, these would be maintained in the Databricks Feature Store.

# Define a "training cutoff" for point-in-time correctness
# We'll use data up to 2025-12-31 for features, hold out 2026+ for validation
TRAINING_CUTOFF = "2025-12-31"

df_views_train = df_views.filter(F.col("view_timestamp") <= TRAINING_CUTOFF)
df_views_test = df_views.filter(F.col("view_timestamp") > TRAINING_CUTOFF)

print(f"Training views (before {TRAINING_CUTOFF}): {df_views_train.count():,}")
print(f"Test views (after {TRAINING_CUTOFF}):      {df_views_test.count():,}")

# Enrich views with content metadata for feature engineering
df_views_enriched = df_views_train.join(df_content, "content_id")

# Step 1: Core user engagement features
df_user_features = (
    df_views_enriched
    .groupBy("user_id")
    .agg(
        # Engagement metrics
        F.count("*").alias("total_views"),
        F.countDistinct("content_id").alias("unique_content_viewed"),
        F.avg("watch_pct").alias("avg_watch_completion"),
        F.avg("rating").alias("avg_rating_given"),
        F.count(F.when(F.col("rating").isNotNull(), 1)).alias("num_ratings"),
        
        # Recency
        F.max("view_timestamp").alias("last_view_date"),
        F.datediff(F.lit(TRAINING_CUTOFF), F.max("view_timestamp")).alias("days_since_last_view"),
        
        # Genre diversity (how many distinct genres does the user watch?)
        F.countDistinct("genre").alias("genre_diversity")
    )
)

# Step 2: Binge indicator - max views per user per day
df_daily_views = (
    df_views_train
    .withColumn("view_date", F.to_date("view_timestamp"))
    .groupBy("user_id", "view_date")
    .agg(F.count("*").alias("daily_views"))
)

df_binge = (
    df_daily_views
    .groupBy("user_id")
    .agg(F.max("daily_views").alias("max_daily_views"))
)

# Step 3: Primary device (most frequently used device per user)
df_device_mode = (
    df_views_train
    .groupBy("user_id", "device_type")
    .agg(F.count("*").alias("device_count"))
)

w = Window.partitionBy("user_id").orderBy(F.desc("device_count"))
df_primary_device = (
    df_device_mode
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .select("user_id", F.col("device_type").alias("primary_device"))
)

# Combine all user features
df_user_features = (
    df_user_features
    .join(df_binge, "user_id", "left")
    .join(df_primary_device, "user_id", "left")
    .join(df_users.select("user_id", "subscription_tier", "region", "age"), "user_id")
)

print(f"\n\u2705 User features created for {df_user_features.count()} users")
display(df_user_features.limit(5))
```

**Why these values?**
* `"watch_pct"` — This column in the viewing history table records what percentage of the content the user actually watched (0–100). Averaging it gives us a measure of how engaged each user is.
* `F.avg` — We want the mean rating across all of a user's rated content. `F.avg` is PySpark's built-in aggregate function for computing averages.
* `F.max` — The most recent view timestamp tells us when the user was last active. `F.max` returns the maximum (latest) value from the `view_timestamp` column.

## Exercise 2 — Point-in-Time Training Dataset

Copy the complete cell below into the Exercise 2 cell:

```python
# --- Point-in-Time Correct Training Dataset ---
# For collaborative filtering, our core input is the user-item interaction matrix.
# We use ONLY training-period data to prevent leakage.
#
# KEY CONCEPT: Point-in-time correctness means:
# - Features for user U are computed from views BEFORE the training cutoff
# - The target (rating/implicit feedback) is also from the training period
# - Test data (after cutoff) is reserved for evaluation only

df_interactions = (
    df_views_train
    .filter(F.col("rating").isNotNull())       # ALS explicit feedback needs ratings
    .groupBy("user_id", "content_id")           # Group by user and content
    .agg(
        F.avg("rating").alias("rating"),         # Average if multiple views
        F.count("*").alias("view_count"),
        F.avg("watch_pct").alias("avg_watch_pct")
    )
)

print(f"Training Interaction Matrix:")
print(f"Unique user-content pairs: {df_interactions.count():,}")
print(f"Unique users:             {df_interactions.select('user_id').distinct().count():,}")
print(f"Unique content items:     {df_interactions.select('content_id').distinct().count():,}")
print(f"Sparsity:                 {1 - df_interactions.count() / (500 * 200):.2%}")

# Save feature tables as Delta for reproducibility
df_user_features.write.mode("overwrite").saveAsTable(f"{SCHEMA}.user_features")
df_content_features.write.mode("overwrite").saveAsTable(f"{SCHEMA}.content_features")
df_interactions.write.mode("overwrite").saveAsTable(f"{SCHEMA}.training_interactions")

print(f"Feature tables saved to {SCHEMA}")
print("  \u2192 user_features")
print("  \u2192 content_features")
print("  \u2192 training_interactions")

display(df_interactions.limit(10))
```

**Why these values?**
* `"rating"` — ALS uses explicit feedback (ratings) to learn preferences. We filter out null ratings because ALS cannot learn from missing values.
* `"user_id", "content_id"` — The interaction matrix is a user-item matrix. Each row represents one unique user-content pair, so we group by both ID columns.
* `"rating"` (in avg) — When a user watches the same content multiple times, we average their ratings to get a single representative score per user-content pair.

## Exercise 3 — Configure ALS Model

Copy the complete cell below into the Exercise 3 cell:

```python
# --- Hyperparameter Tuning with MLflow ---
# We'll run a grid search, logging each combination to MLflow

from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

# Set up MLflow experiment
mlflow.set_experiment(EXPERIMENT_NAME)

# Define hyperparameter grid
param_grid = {
    "rank": [10, 50, 100],
    "maxIter": [10, 15],
    "regParam": [0.01, 0.1, 0.5]
}

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

best_rmse = float("inf")
best_model = None
best_params = {}
run_count = 0
total_runs = len(param_grid["rank"]) * len(param_grid["maxIter"]) * len(param_grid["regParam"])

print(f"Starting hyperparameter search: {total_runs} combinations")
print("=" * 60)

for rank in param_grid["rank"]:
    for maxIter in param_grid["maxIter"]:
        for regParam in param_grid["regParam"]:
            run_count += 1
            run_name = f"ALS_r{rank}_i{maxIter}_reg{regParam}"
            
            with mlflow.start_run(run_name=run_name) as run:
                # Log parameters
                mlflow.log_param("rank", rank)
                mlflow.log_param("maxIter", maxIter)
                mlflow.log_param("regParam", regParam)
                mlflow.log_param("algorithm", "ALS")
                mlflow.log_param("feedback_type", "explicit")
                
                # Train ALS model
                als = ALS(
                    rank=rank,
                    maxIter=maxIter,
                    regParam=regParam,
                    userCol="user_id",
                    itemCol="content_id",
                    ratingCol="rating",
                    coldStartStrategy="drop",
                    nonnegative=True
                )
                
                model = als.fit(als_train)
                
                # Evaluate on validation set
                predictions = model.transform(als_val)
                rmse = evaluator.evaluate(predictions)
                
                # Log metrics
                mlflow.log_metric("rmse", rmse)
                mlflow.log_metric("num_user_factors", model.userFactors.count())
                mlflow.log_metric("num_item_factors", model.itemFactors.count())
                
                # Track best model
                if rmse < best_rmse:
                    best_rmse = rmse
                    best_model = model
                    best_params = {"rank": rank, "maxIter": maxIter, "regParam": regParam}
                    best_run_id = run.info.run_id
                
                status = " \u2b50 BEST" if rmse == best_rmse else ""
                print(f"  [{run_count}/{total_runs}] {run_name}: RMSE={rmse:.4f}{status}")

print("\n" + "=" * 60)
print(f"Best Model: RMSE={best_rmse:.4f}")
print(f" Parameters: {best_params}")
print(f"Run ID: {best_run_id}")
```

**Why these values?**
* `"user_id"` — This is the column name in our interaction matrix that identifies users. ALS needs to know which column to use for the row dimension of the user-item matrix.
* `"content_id"` — This is the column name that identifies content items. ALS uses this for the column dimension of the user-item matrix.
* `"rating"` — This is the column containing the explicit feedback values (1–5 star ratings) that ALS will try to predict. It must match the column name we created in Exercise 2.

## Exercise 4 — Register Model to Unity Catalog

Copy the complete cell below into the Exercise 4 cell:

```python
# The ALS model is logged to UC for versioning, lineage, and reproducibility.
# Serving is handled by Feature Serving (Part 3), not by a model serving endpoint.
from mlflow.models import infer_signature

mlflow.set_registry_uri("databricks-uc")

# UC Volume for temporary Spark model storage (required on serverless)
VOLUME_PATH = f"/Volumes/content_reco_lab/content_reco_demo/ml_artifacts"
spark.sql(f"CREATE VOLUME IF NOT EXISTS {SCHEMA}.ml_artifacts")

sample_input = als_train.select("user_id", "content_id").limit(5).toPandas()

with mlflow.start_run(run_name="best_als_model_registration") as run:
    mlflow.log_params(best_params)
    mlflow.log_metric("rmse", best_rmse)

    mlflow.spark.log_model(
        spark_model=best_model,
        artifact_path="als_model",
        registered_model_name=MODEL_NAME,
        input_example=sample_input,
        dfs_tmpdir=f"{VOLUME_PATH}/tmp",
    )

    print(f"\n\u2705 Model registered to Unity Catalog: {MODEL_NAME}")
    print(f"   Run ID: {run.info.run_id}")

from mlflow import MlflowClient
client = MlflowClient()
latest_versions = client.search_model_versions(f"name='{MODEL_NAME}'")
if latest_versions:
    latest_version = max(int(v.version) for v in latest_versions)
    client.set_registered_model_alias(MODEL_NAME, "Champion", str(latest_version))
    print(f"   Alias 'Champion' set to version {latest_version}")
```

**Why these values?**
* `"als_model"` — This is the artifact path (subfolder name) under which the model files are stored within the MLflow run. It's a descriptive label — any meaningful string works.
* `MODEL_NAME` — This variable (defined in the Configuration cell as `content_reco_lab.content_reco_demo.content_recommender_als`) is the fully qualified Unity Catalog path where the model is registered. Using the variable ensures consistency.
* `"Champion"` — This is a standard alias convention in MLflow for marking the production-ready model version. Downstream systems reference the alias instead of a version number, making model promotion seamless.

## Exercise 5 — Create Feature Spec

Copy the complete cell below into the Exercise 5 cell:

```python
# --- Step 3: Create Feature Spec ---

print("\nStep 3: Creating feature spec...")
features = [
    FeatureLookup(
        table_name=f"{SCHEMA}.serving_user_recs",
        lookup_key="user_id",
    ),
    FeatureLookup(
        table_name=f"{SCHEMA}.serving_item_similarities",
        lookup_key="content_id",
    ),
]

try:
    fe.create_feature_spec(name=FEATURE_SPEC_NAME, features=features)
    print(f"  \u2705 Created: {FEATURE_SPEC_NAME}")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"  \u2139\ufe0f  Already exists: {FEATURE_SPEC_NAME}")
    else:
        raise
```

**Why these values?**
* `serving_user_recs` — This is the wide-format Delta table we built in Step 4.2 containing one row per user with their top-10 personalized recommendations.
* `"user_id"` — This is the primary key of the user recs table. When the endpoint receives a `user_id`, the Feature Spec tells the serving layer to look up that user's row.
* `serving_item_similarities` — This is the wide-format Delta table built in Step 4.1 containing one row per content item with its top-10 similar items.
* `"content_id"` — This is the primary key of the item similarities table. It maps a content item to its most similar items.

## Exercise 6 — Test the Serving Endpoint

Copy the complete cell below into the Exercise 6 cell:

```python
# --- Test Feature Serving Endpoint ---
# Scenario: A user just finished watching a show.
# The app calls the endpoint with {user_id, content_id} and gets:
#   \u2022 "Top 10 For You" \u2014 personalized recommendations
#   \u2022 "Because You Watched" \u2014 items similar to what they watched

import mlflow.deployments

client = mlflow.deployments.get_deploy_client("databricks")

response = client.predict(
    endpoint=ENDPOINT_NAME,
    inputs={
        "dataframe_records": [
            {"user_id": 42, "content_id": 147}
        ]
    },
)

result = response["outputs"][0]

print("\U0001f3ac User 42 just watched content 147")
print("=" * 60)

# Display personalized recommendations
print("\n\U0001f4cb Top 10 For You (personalized):")
for i in range(1, 11):
    title = result.get(f"rec_{i}_title", "\u2014")
    genre = result.get(f"rec_{i}_genre", "\u2014")
    score = result.get(f"rec_{i}_score", 0)
    if title != "\u2014":
        print(f"  {i:>2}. [{genre}] {title} (score: {score})")

# Display similar items
print(f"\n\U0001f517 Because You Watched Content 147:")
for i in range(1, 11):
    title = result.get(f"sim_{i}_title", "\u2014")
    genre = result.get(f"sim_{i}_genre", "\u2014")
    score = result.get(f"sim_{i}_score", 0)
    if title != "\u2014":
        print(f"  {i:>2}. [{genre}] {title} (similarity: {score})")

print(f"\n\u2705 Feature Serving: <10ms latency, no model inference \u2014 just lookups!")
```

**Why these values?**
* `ENDPOINT_NAME` — This variable (defined as `"content-recommender-endpoint"`) references the Feature Serving endpoint we created in Step 4.5. Using the variable keeps the code DRY.
* `"user_id"` and `"content_id"` — These are the lookup keys defined in the Feature Spec (Exercise 5). The endpoint expects a dictionary with these exact key names so it knows which table to query for each value. The values `42` and `147` are example IDs — any valid user ID (1–500) and content ID (1–200) would work.